# Deep Learning: A Comprehensive Guide

## Chapter 5 — Advanced CNN Architectures

### Hands-On Exploration: From Classification to Detection

### `hands_on_ch5.ipynb`

This activity builds directly on Chapter 4's Architecture Archaeology exercise. You already have a fine-tuned ResNet-50 that classifies images. This week, you will see what that same backbone is "seeing" at a spatial level, and then load a pre-trained detection model to observe how detection heads interpret backbone features differently from a classification head.

## Learning Objectives

By the end of this notebook, you should be able to:

- Compare a classifier's spatial activation pattern to a detector's bounding-box outputs on the same image
- Interpret how confidence thresholds trade off precision and recall in an object detector
- Compare instance segmentation masks to ground-truth object boundaries and identify sources of error

## Prerequisites

- Python 3.9+, `numpy`, `matplotlib`, `Pillow`
- `torch`, `torchvision` (for Mask R-CNN) and `ultralytics` (for YOLOv8n) — install with:
  ```
  pip install torch torchvision ultralytics
  ```
- **This notebook is designed to run in Google Colab**, exactly as the chapter specifies. Both `ultralytics` and `torchvision`'s pretrained weights are downloaded from public model hosts the first time each model is used, which requires internet access. If you are working in a fully offline environment, the notebook detects this and falls back to a lightweight synthetic demonstration so every cell still runs — but see the note below.
- No GPU required — YOLOv8 nano and Mask R-CNN inference both run adequately on CPU for a handful of images

## Setup — Imports, Reproducibility, and Model Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

SEED = 42
np.random.seed(SEED)

%matplotlib inline

TORCH_AVAILABLE = False
ULTRALYTICS_AVAILABLE = False

try:
    import torch
    import torchvision
    TORCH_AVAILABLE = True
except ImportError:
    print("PyTorch/torchvision not installed. Install with: pip install torch torchvision")

try:
    from ultralytics import YOLO
    ULTRALYTICS_AVAILABLE = True
except ImportError:
    print("ultralytics not installed. Install with: pip install ultralytics")


## Generate Test Scenes

We generate three synthetic scenes matching the chapter's Part 1 description: (a) one clear subject centered in the frame, (b) multiple objects of the same class, (c) multiple objects of different classes. If you have your own fine-tuned ResNet-50 from Chapter 4 or real photographs, substitute them here.

In [ ]:
IMG_SIZE = 256


def draw_blob(draw, cx, cy, r, color):
    draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)


def make_scene(kind, seed):
    rng = np.random.RandomState(seed)
    img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(235, 235, 230))
    draw = ImageDraw.Draw(img)

    objects = []  # (class_name, cx, cy, r)
    if kind == "single_centered":
        objects = [("dog", IMG_SIZE // 2, IMG_SIZE // 2, 40)]
    elif kind == "multi_same_class":
        objects = [
            ("person", 70, 90, 30), ("person", 170, 100, 28), ("person", 120, 190, 32),
        ]
    elif kind == "multi_diff_class":
        objects = [
            ("dog", 70, 90, 30), ("car", 180, 160, 35), ("bird", 130, 60, 18),
        ]

    colors = {"dog": (200, 150, 80), "person": (120, 90, 200), "car": (80, 150, 200), "bird": (200, 80, 80)}
    for cls, cx, cy, r in objects:
        draw_blob(draw, cx, cy, r, colors[cls])

    return np.array(img).astype(np.float32) / 255.0, objects


scenes = {
    "single_centered": make_scene("single_centered", 1),
    "multi_same_class": make_scene("multi_same_class", 2),
    "multi_diff_class": make_scene("multi_diff_class", 3),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, (img, objs)) in zip(axes, scenes.items()):
    ax.imshow(img); ax.set_title(name.replace("_", " ")); ax.axis("off")
plt.tight_layout()
plt.show()


## Part 1 — Spatial Feature Visualization

Using the GradCAM tool from Chapter 4, generate activation maps for the three scenes above using your fine-tuned (or ImageNet-pretrained) ResNet-50.

In [ ]:
if TORCH_AVAILABLE:
    from torchvision.models import resnet50, ResNet50_Weights

    try:
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        print("\u2713 Loaded ImageNet-pretrained ResNet-50 (or your fine-tuned Chapter 4 checkpoint if you swap it in).")
    except Exception as e:
        resnet = resnet50(weights=None)
        print(f"Could not download pretrained weights ({type(e).__name__}); using an untrained ResNet-50.")
        print("Run this notebook in Google Colab for genuine pretrained behavior.")
    resnet.eval()

    def resnet_gradcam(model, img_np, target_layer):
        img_t = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float()
        activations, gradients = [], []

        def fwd_hook(module, inp, out):
            activations.append(out)

        def bwd_hook(module, grad_in, grad_out):
            gradients.append(grad_out[0])

        h1 = target_layer.register_forward_hook(fwd_hook)
        h2 = target_layer.register_full_backward_hook(bwd_hook)

        output = model(img_t)
        class_idx = output.argmax(dim=1).item()
        model.zero_grad()
        output[0, class_idx].backward()

        h1.remove(); h2.remove()

        acts = activations[0][0]
        grads = gradients[0][0]
        weights = grads.mean(dim=(1, 2))
        cam = torch.relu((weights[:, None, None] * acts).sum(dim=0))
        cam = cam / (cam.max() + 1e-8)
        return cam.detach().numpy(), class_idx

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for col, (name, (img, objs)) in enumerate(scenes.items()):
        cam, cls_idx = resnet_gradcam(resnet, img, resnet.layer4[-1])
        axes[0, col].imshow(img); axes[0, col].set_title(name); axes[0, col].axis("off")
        axes[1, col].imshow(img)
        cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))
        axes[1, col].imshow(cam_resized, cmap="jet", alpha=0.5)
        axes[1, col].set_title(f"GradCAM (predicted class {cls_idx})"); axes[1, col].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("PyTorch not available -- install torch/torchvision to run Part 1.")


**Answer in this cell:** Does the classification model's activation spread across all objects of the target class, or focus on one? What happens when you ask it to classify a class that appears in the background? Does the activation shift to the background region?

## Part 2 — YOLO Detection Walkthrough

Run the pre-loaded YOLOv8 nano model on the same three scenes and visualize the output bounding boxes and confidence scores.

In [ ]:
def draw_boxes(ax, img, boxes, labels, scores, threshold):
    ax.imshow(img)
    for box, label, score in zip(boxes, labels, scores):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 4, 0), f"{label} {score:.2f}", color="lime", fontsize=8,
                 bbox=dict(facecolor="black", alpha=0.5, pad=1))
    ax.axis("off")


if ULTRALYTICS_AVAILABLE:
    try:
        yolo = YOLO("yolov8n.pt")  # downloads the nano checkpoint on first use
        YOLO_LOADED = True
    except Exception as e:
        print(f"Could not download YOLOv8n weights ({type(e).__name__}). Run this notebook in Colab.")
        YOLO_LOADED = False

    if YOLO_LOADED:
        for threshold in [0.5, 0.2, 0.8]:
            fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
            fig.suptitle(f"YOLOv8n detections at confidence threshold = {threshold}")
            for ax, (name, (img, objs)) in zip(axes, scenes.items()):
                results = yolo.predict((img * 255).astype(np.uint8), verbose=False)[0]
                boxes = results.boxes.xyxy.numpy() if len(results.boxes) else np.empty((0, 4))
                scores = results.boxes.conf.numpy() if len(results.boxes) else np.empty((0,))
                labels = [results.names[int(c)] for c in results.boxes.cls] if len(results.boxes) else []
                draw_boxes(ax, img, boxes, labels, scores, threshold)
                ax.set_title(name)
            plt.tight_layout()
            plt.show()
else:
    print("ultralytics not available -- install it with `pip install ultralytics` to run Part 2.")
    print("This notebook's synthetic scenes are simple colored blobs, not photographs, so a real")
    print("object detector trained on COCO will likely detect zero or spurious objects on them --")
    print("substitute real photographs here for a meaningful detection walkthrough.")


**Answer in this cell:** How many objects does YOLO find compared to what the classifier saw? What is the lowest-confidence detection? What new detections appear at threshold 0.2 that were absent at 0.5, and what do they tell you about the model's uncertainty? What disappears at threshold 0.8?

## Part 3 — Instance Segmentation

Run a pre-loaded Mask R-CNN model on the `multi_same_class` scene (two or more objects of the same class) and visualize the output instance masks.

In [ ]:
if TORCH_AVAILABLE:
    from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

    try:
        mask_rcnn = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
        MASKRCNN_LOADED = True
    except Exception as e:
        print(f"Could not download Mask R-CNN weights ({type(e).__name__}). Run this notebook in Colab.")
        MASKRCNN_LOADED = False

    if MASKRCNN_LOADED:
        mask_rcnn.eval()
        img, objs = scenes["multi_same_class"]
        img_t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float()
        with torch.no_grad():
            output = mask_rcnn(img_t)[0]

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(img)
        for mask, score in zip(output["masks"], output["scores"]):
            if score < 0.5:
                continue
            mask_np = mask[0].numpy()
            ax.imshow(np.ma.masked_where(mask_np < 0.5, mask_np), cmap="autumn", alpha=0.5)
        ax.set_title("Mask R-CNN instance masks (score > 0.5)")
        ax.axis("off")
        plt.show()
else:
    print("PyTorch not available -- install torch/torchvision to run Part 3.")


**Answer in this cell:** Does Mask R-CNN successfully distinguish the two instances? How precise are the mask boundaries? Find a region where the mask and the actual object boundary diverge — what spatial structure seems to be causing the error?

## Reflection

Write three sentences:

1. What does this exercise reveal about the difference between what a classifier knows about an image and what a detector knows?
2. How does varying the confidence threshold illustrate the precision-recall tradeoff in practice?
3. If you were deploying a detection system in a context where false positives are more costly than false negatives (for example, a triage system that triggers costly follow-up procedures), how would you adjust the threshold and what would you monitor?

**Your reflection (edit this cell):**

1. _..._
2. _..._
3. _..._